# Evaluation for Pitch Keypoints and Homography Transformation

In [ ]:
!git clone https://github.comgithub.com/EkmalRey/FutsalCV.git
# Optional installs (uncomment if missing deps)
%pip install -q huggingface_hub ultralytics opencv-python matplotlib tqdm supervision

## Import Libraries

In [ ]:
from pathlib import Path
import json
import random
import sys
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

# Ensure helper modules resolve
# Notebook is in Models/Evaluation
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]

# Add paths
paths_to_add = [
    PROJECT_ROOT, # For config
    PROJECT_ROOT / "Data", # For _1_... and _3_...
    PROJECT_ROOT / "Models" / "Training", # For train_keypoints_helper
    NOTEBOOK_DIR # For evaluation_object_helper (current dir)
]

for p in paths_to_add:
    if str(p) not in sys.path:
        sys.path.append(str(p))

from config import PoseEstimationConfig, get_device
from _1_download_dataset import download_dataset
from _3_0_convert_raw_to_yolo_keypoints import (
    ConvertConfig,
    KEYPOINT_ORDER,
    _create_dataset_yaml,
    process_split,
)
from train_keypoints_helper import evaluate_keypoint_metrics
from evaluation_object_helper import reprojection_error

#### CONFIG

In [ ]:
# Paths and constants
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]
RESOURCES_DIR = PROJECT_ROOT / "Resources"
DATA_ROOT = RESOURCES_DIR / "Dataset" / "SN-GSR-2025"
YOLO_ROOT = RESOURCES_DIR / "Dataset" / "yolo_soccernet_pitch"
RESULTS_DIR = RESOURCES_DIR / "Models"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "keypoints_eval_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model
FIELD_MODEL_PATH = RESULTS_DIR / "Fields.pt"

# Evaluation settings
SPLIT = "test"
SAMPLE_VIS_COUNT = 6
CONFIDENCE = 0.5
DEVICE = get_device()

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"YOLO root:    {YOLO_ROOT}")
print(f"Model path:   {FIELD_MODEL_PATH}")
print(f"Device:       {DEVICE}")

## Download & Load Dataset

In [ ]:
def ensure_yolo_dataset() -> Tuple[Path, Path]:
    base_output = YOLO_ROOT / "soccernet_pitch"
    dataset_yaml = base_output / "dataset.yaml"
    img_test_dir = base_output / "images" / SPLIT
    lbl_test_dir = base_output / "labels" / SPLIT

    def _has_files(path: Path, exts=("*.jpg", "*.png", "*.jpeg")) -> bool:
        return any(path.glob(pattern) for pattern in exts)

    if dataset_yaml.exists() and img_test_dir.exists() and lbl_test_dir.exists() and _has_files(img_test_dir) and _has_files(lbl_test_dir, ("*.txt",)):
        print("✅ YOLO dataset already present (test split)")
        return dataset_yaml, base_output

    print("⏬ Preparing dataset (test split only)...")
    cfg = PoseEstimationConfig(data_root=DATA_ROOT, yolo_root=YOLO_ROOT)
    download_dataset(cfg, debug_test_only=True)

    base_output.mkdir(parents=True, exist_ok=True)
    convert_cfg = ConvertConfig(
        data_root=cfg.data_root,
        yolo_root=YOLO_ROOT,
        splits=[SPLIT],
        max_games=None,
        max_images=None,
        overwrite=False,
        save_unified_json=False,
        convert_workers=None,
    )

    stats = process_split(convert_cfg, SPLIT, base_output)
    dataset_yaml = _create_dataset_yaml(base_output)

    # Ensure empty train/val dirs exist so YOLO yaml paths are valid even when unused
    for split_name in ("train", "val"):
        (base_output / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (base_output / "labels" / split_name).mkdir(parents=True, exist_ok=True)

    if not _has_files(img_test_dir) or not _has_files(lbl_test_dir, ("*.txt",)):
        raise RuntimeError("Conversion did not produce test images/labels.")

    print(f"Conversion stats (test): {stats}")
    print(f"Dataset yaml: {dataset_yaml}")
    return dataset_yaml, base_output


data_yaml_path, yolo_data_root = ensure_yolo_dataset()

## Process Dataset (Convert to YOLO Format)

In [ ]:
def summarize_split(base: Path, split: str) -> Dict[str, int]:
    imgs = list((base / "images" / split).glob("*.jpg")) + list((base / "images" / split).glob("*.png"))
    lbls = list((base / "labels" / split).glob("*.txt"))
    return {"images": len(imgs), "labels": len(lbls)}

summary_test = summarize_split(yolo_data_root, SPLIT)
print(f"Test split: {summary_test}")
print(f"Data yaml -> {data_yaml_path}")

## Load Model

In [ ]:
if not FIELD_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model at {FIELD_MODEL_PATH}")

field_model = YOLO(str(FIELD_MODEL_PATH))
print(f"Loaded field model from {FIELD_MODEL_PATH}")
print(f"Model names: {field_model.model.names}")

## Load Pitch Configuration

In [ ]:
# Pitch configuration (29-keypoint SoccerNet layout)
KEYPOINT_NAMES = {
    1: "sideline_top_left",
    2: "big_rect_left_top_pt1",
    3: "big_rect_left_top_pt2",
    4: "big_rect_left_bottom_pt1",
    5: "big_rect_left_bottom_pt2",
    6: "small_rect_left_top_pt1",
    7: "small_rect_left_top_pt2",
    8: "small_rect_left_bottom_pt1",
    9: "small_rect_left_bottom_pt2",
    10: "sideline_bottom_left",
    11: "left_semicircle_right",
    12: "center_line_top",
    13: "center_line_bottom",
    14: "center_circle_top",
    15: "center_circle_bottom",
    16: "field_center",
    17: "sideline_top_right",
    18: "big_rect_right_top_pt1",
    19: "big_rect_right_top_pt2",
    20: "big_rect_right_bottom_pt1",
    21: "big_rect_right_bottom_pt2",
    22: "small_rect_right_top_pt1",
    23: "small_rect_right_top_pt2",
    24: "small_rect_right_bottom_pt1",
    25: "small_rect_right_bottom_pt2",
    26: "sideline_bottom_right",
    27: "right_semicircle_left",
    28: "center_circle_left",
    29: "center_circle_right",
}

MINIMAP_SCALE = 0.1
MINIMAP_PADDING = 50
MINIMAP_KEYPOINT_COLOR = (255, 255, 255)

@dataclass
class SoccerPitchConfiguration:
    length: int = 12000
    width: int = 7000
    penalty_box_length: int = 1886
    penalty_box_width: int = 4140
    goal_box_length: int = 629
    goal_box_width: int = 1885
    centre_circle_radius: int = 942
    penalty_spot_distance: int = 1257

    @property
    def vertices(self) -> List[Tuple[int, int]]:
        top_penalty = (self.width - self.penalty_box_width) / 2
        bottom_penalty = self.width - top_penalty
        top_goal = (self.width - self.goal_box_width) / 2
        bottom_goal = self.width - top_goal
        center_y = self.width / 2
        arc_radius = self.centre_circle_radius
        penalty_spot = self.penalty_spot_distance
        return [
            (0, 0),
            (0, top_penalty),
            (self.penalty_box_length, top_penalty),
            (0, bottom_penalty),
            (self.penalty_box_length, bottom_penalty),
            (0, top_goal),
            (self.goal_box_length, top_goal),
            (0, bottom_goal),
            (self.goal_box_length, bottom_goal),
            (0, self.width),
            (penalty_spot + arc_radius, center_y),
            (self.length / 2, 0),
            (self.length / 2, self.width),
            (self.length / 2, center_y - arc_radius),
            (self.length / 2, center_y + arc_radius),
            (self.length / 2, center_y),
            (self.length, 0),
            (self.length, top_penalty),
            (self.length - self.penalty_box_length, top_penalty),
            (self.length, bottom_penalty),
            (self.length - self.penalty_box_length, bottom_penalty),
            (self.length, top_goal),
            (self.length - self.goal_box_length, top_goal),
            (self.length, bottom_goal),
            (self.length - self.goal_box_length, bottom_goal),
            (self.length, self.width),
            (self.length - (penalty_spot + arc_radius), center_y),
            (self.length / 2 - arc_radius, center_y),
            (self.length / 2 + arc_radius, center_y),
        ]

    edges: List[Tuple[int, int]] = None
    line_edges: List[Tuple[int, int]] = None

    def __post_init__(self) -> None:
        self.edges = [
            (1, 17), (1, 10), (17, 26), (10, 26),
            (2, 3), (4, 5), (2, 4), (3, 5),
            (6, 7), (8, 9), (6, 8), (7, 9),
            (18, 19), (20, 21), (18, 20), (19, 21),
            (22, 23), (24, 25), (22, 24), (23, 25),
            (12, 13),
        ]
        self.line_edges = [
            (1, 10), (1, 17), (17, 26), (10, 26),
            (2, 3), (2, 4), (3, 5), (4, 5),
            (6, 7), (6, 8), (7, 9), (8, 9),
            (18, 19), (18, 20), (19, 21), (20, 21),
            (22, 23), (22, 24), (23, 25), (24, 25),
            (12, 14), (13, 15), (14, 16), (15, 16),
            (14, 28), (14, 29), (15, 28), (15, 29), (28, 29),
        ]


def draw_pitch(config: SoccerPitchConfiguration, scale: float = MINIMAP_SCALE, padding: int = MINIMAP_PADDING) -> np.ndarray:
    import math

    line_color = (255, 255, 255)
    background = (34, 139, 34)
    scaled_w = int(config.width * scale)
    scaled_l = int(config.length * scale)
    scaled_circle = int(config.centre_circle_radius * scale)
    scaled_spot = int(config.penalty_spot_distance * scale)

    canvas = np.ones((scaled_w + 2 * padding, scaled_l + 2 * padding, 3), dtype=np.uint8)
    canvas[:] = background

    for start, end in config.edges:
        p1 = (int(config.vertices[start - 1][0] * scale) + padding, int(config.vertices[start - 1][1] * scale) + padding)
        p2 = (int(config.vertices[end - 1][0] * scale) + padding, int(config.vertices[end - 1][1] * scale) + padding)
        cv2.line(canvas, p1, p2, line_color, 4)

    center = (scaled_l // 2 + padding, scaled_w // 2 + padding)
    cv2.circle(canvas, center, scaled_circle, line_color, 4)

    penalty_pts = [
        (scaled_spot + padding, scaled_w // 2 + padding),
        (scaled_l - scaled_spot + padding, scaled_w // 2 + padding),
    ]
    for spot in penalty_pts:
        cv2.circle(canvas, spot, 8, line_color, -1)

    offset = (config.penalty_box_length - config.penalty_spot_distance) * scale
    clamped = min(max(offset / max(scaled_circle, 1e-6), -1.0), 1.0)
    arc_angle = math.degrees(math.acos(clamped))

    left_center = (scaled_spot + padding, scaled_w // 2 + padding)
    right_center = (scaled_l - scaled_spot + padding, scaled_w // 2 + padding)

    cv2.ellipse(canvas, left_center, (scaled_circle, scaled_circle), 0, -arc_angle, arc_angle, line_color, 4)
    cv2.ellipse(canvas, right_center, (scaled_circle, scaled_circle), 0, 180 - arc_angle, 180 + arc_angle, line_color, 4)
    return canvas


class ViewTransformer:
    def __init__(self, source: np.ndarray, target: np.ndarray) -> None:
        if source.shape != target.shape or source.shape[1] != 2:
            raise ValueError("Source and target must be Nx2 arrays")
        m, _ = cv2.findHomography(source.astype(np.float32), target.astype(np.float32))
        if m is None:
            raise ValueError("Homography matrix could not be computed")
        self.m = m

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points
        reshaped = points.reshape(-1, 1, 2).astype(np.float32)
        transformed = cv2.perspectiveTransform(reshaped, self.m)
        return transformed.reshape(-1, 2).astype(np.float32)


def detect_field_keypoints(frame: np.ndarray, model: YOLO, config: SoccerPitchConfiguration, confidence: float = CONFIDENCE) -> Tuple[np.ndarray, np.ndarray]:
    result = model(frame, conf=confidence, verbose=False)[0]
    kps = getattr(result, "keypoints", None)
    if kps is None or kps.xy is None:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)

    xy = kps.xy
    conf_arr = getattr(kps, "conf", None)
    xy_np = xy[0].cpu().numpy() if hasattr(xy, "cpu") else np.asarray(xy)[0]
    conf_np = conf_arr[0].cpu().numpy() if (conf_arr is not None and hasattr(conf_arr, "cpu")) else (np.asarray(conf_arr)[0] if conf_arr is not None else None)

    expected = len(config.vertices)
    xy_np = xy_np[:expected]
    if conf_np is not None:
        conf_np = conf_np[:expected]

    if xy_np.size == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)

    mask = conf_np > confidence if conf_np is not None else np.ones(len(xy_np), dtype=bool)
    if not mask.any():
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)

    pitch_vertices = np.array(config.vertices, dtype=np.float32)[: len(xy_np)]
    frame_points = xy_np[mask].astype(np.float32)
    pitch_points = pitch_vertices[mask]
    return frame_points, pitch_points


def compute_view_transformers(frame: np.ndarray, model: YOLO, config: SoccerPitchConfiguration, confidence: float = CONFIDENCE) -> Tuple[Optional[ViewTransformer], Optional[ViewTransformer], np.ndarray, np.ndarray]:
    frame_points, pitch_points = detect_field_keypoints(frame, model, config, confidence)
    if len(frame_points) < 4 or len(pitch_points) < 4:
        return None, None, frame_points, pitch_points
    forward = ViewTransformer(source=pitch_points, target=frame_points)
    inverse = ViewTransformer(source=frame_points, target=pitch_points)
    return forward, inverse, frame_points, pitch_points


def minimap_coords(x: float, y: float, scale: float = MINIMAP_SCALE, padding: int = MINIMAP_PADDING) -> Tuple[int, int]:
    return (int(x * scale) + padding, int(y * scale) + padding)


PITCH_CONFIG = SoccerPitchConfiguration()
BASE_MINIMAP = draw_pitch(PITCH_CONFIG)
print(f"Pitch vertices: {len(PITCH_CONFIG.vertices)} keypoints")

## Detect and Project Homography (including Evaluation for Reprojection Error and Mean Reprojection Error)

In [ ]:
# Evaluate keypoints on the full test split
print("🚀 Running keypoint evaluation on test split...")
metrics_test = evaluate_keypoint_metrics(
    model=field_model,
    yolo_data_root=yolo_data_root,
    split=SPLIT,
    conf_threshold=CONFIDENCE,
    device=str(DEVICE),
    keypoint_names=KEYPOINT_ORDER,
    verbose=True,
)
metrics_path = OUTPUT_DIR / "keypoint_metrics_test.json"
metrics_path.write_text(json.dumps(metrics_test, indent=2))
print(f"✅ Saved keypoint metrics -> {metrics_path}")

# Reprojection error over the entire test split
print("\n🧮 Computing homography reprojection error on ALL test images...")
all_images = sorted((yolo_data_root / "images" / SPLIT).glob("*.jpg"))
if not all_images:
    all_images = sorted((yolo_data_root / "images" / SPLIT).glob("*.png"))
if not all_images:
    raise FileNotFoundError("No test images found for homography evaluation")

reproj_results: List[Dict[str, object]] = []
for img_path in tqdm(all_images, desc="Reprojection (all test)", unit="img"):
    frame = cv2.imread(str(img_path))
    if frame is None:
        reproj_results.append({"image": img_path.name, "status": "read_error"})
        continue

    pitch_to_frame, frame_to_pitch, frame_pts, pitch_pts = compute_view_transformers(frame, field_model, PITCH_CONFIG, confidence=CONFIDENCE)
    if pitch_to_frame is None or frame_to_pitch is None:
        reproj_results.append({"image": img_path.name, "status": "insufficient_keypoints"})
        continue

    errors, mean_err = reprojection_error(pitch_to_frame.m, pitch_pts, frame_pts)
    reproj_results.append({
        "image": img_path.name,
        "status": "ok",
        "mean_reprojection_px": mean_err,
        "num_points": len(errors),
    })

valid_all = [r for r in reproj_results if r.get("status") == "ok"]
mean_reproj_all = float(np.mean([r["mean_reprojection_px"] for r in valid_all])) if valid_all else float("nan")
summary_all = {
    "num_images": len(all_images),
    "num_valid": len(valid_all),
    "mean_reprojection_px": mean_reproj_all,
}
print(f"✅ Homography reprojection done: {summary_all}")

# Sample visualization subset (still limited for plotting)
random.seed(42)
sample_paths = random.sample(all_images, min(SAMPLE_VIS_COUNT, len(all_images)))
print(f"🖼️ Generating overlays for {len(sample_paths)} sampled frames...")

sample_results: List[Dict[str, object]] = []
for idx, img_path in enumerate(sample_paths, start=1):
    frame = cv2.imread(str(img_path))
    if frame is None:
        sample_results.append({"image": img_path.name, "status": "read_error"})
        continue

    pitch_to_frame, frame_to_pitch, frame_pts, pitch_pts = compute_view_transformers(frame, field_model, PITCH_CONFIG, confidence=CONFIDENCE)
    if pitch_to_frame is None or frame_to_pitch is None:
        sample_results.append({"image": img_path.name, "status": "insufficient_keypoints"})
        continue

    errors, mean_err = reprojection_error(pitch_to_frame.m, pitch_pts, frame_pts)
    sample_results.append({
        "image": img_path.name,
        "status": "ok",
        "mean_reprojection_px": mean_err,
        "num_points": len(errors),
    })

    # Overlay detected and template keypoints on the frame
    vis = frame.copy()
    for i, (x, y) in enumerate(frame_pts):
        cv2.circle(vis, (int(x), int(y)), 5, (0, 215, 255), -1)
        cv2.putText(vis, str(i + 1), (int(x) + 4, int(y) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    projected_template = pitch_to_frame.transform_points(pitch_pts)
    for x, y in projected_template:
        cv2.circle(vis, (int(x), int(y)), 4, (0, 0, 0), 1)

    minimap = BASE_MINIMAP.copy()
    detected_pitch = frame_to_pitch.transform_points(frame_pts)
    for (px, py) in detected_pitch:
        cv2.circle(minimap, minimap_coords(px, py), 8, (0, 0, 0), -1)
    for (px, py) in pitch_pts:
        cv2.circle(minimap, minimap_coords(px, py), 6, MINIMAP_KEYPOINT_COLOR, 2)

    out_frame = OUTPUT_DIR / f"sample_{idx:02d}_frame.jpg"
    out_minimap = OUTPUT_DIR / f"sample_{idx:02d}_minimap.jpg"
    cv2.imwrite(str(out_frame), vis)
    cv2.imwrite(str(out_minimap), minimap)

# Aggregate reprojection summaries
valid_samples = [r for r in sample_results if r.get("status") == "ok"]
mean_reproj_samples = float(np.mean([r["mean_reprojection_px"] for r in valid_samples])) if valid_samples else float("nan")

reproj_summary = {
    "all_images_summary": summary_all,
    "samples": sample_results,
    "mean_reprojection_px_samples": mean_reproj_samples,
    "num_valid_samples": len(valid_samples),
    "num_sampled": len(sample_results),
}

reproj_path = OUTPUT_DIR / "reprojection_samples_test.json"
reproj_path.write_text(json.dumps(reproj_summary, indent=2))
print(f"✅ Saved reprojection summary -> {reproj_path}")

## Save Data to File

Results are written under `outputs/keypoints_eval_test/`:
- `keypoint_metrics_test.json`: full test-split keypoint metrics (MKE, PCK, per-kp).
- `reprojection_samples_test.json`: per-sample reprojection stats and mean.
- `sample_XX_frame.jpg` / `sample_XX_minimap.jpg`: visual overlays for the sampled frames.
